# Interactive LU Factorization

Explore LU factorization with partial pivoting: $PA=LU$

The dashboard visualizes the matrix factors and verifies the accuracy of solving $Ax=b$.

In [2]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

root = Path.cwd().resolve()
while not (root / 'direct_methods' / 'lu_factorization' / 'lu.py').exists():
    if root == root.parent:
        raise RuntimeError('Repository root not found.')
    root = root.parent
sys.path.insert(0, str(root))

from direct_methods.lu_factorization import lu_factorization, lu_solve

In [3]:
def show_lu(n=4, seed=42, matrix_type='Random'):
    rng = np.random.default_rng(seed)

    if matrix_type == 'Random':
        A = rng.standard_normal((n, n))
    elif matrix_type == 'Symmetric positive definite':
        B = rng.standard_normal((n, n))
        A = B.T @ B + np.eye(n)
    else:
        A = rng.standard_normal((n, n))
        A[0, 0] = 0.0
        A[1, 0] = 2.0

    b = A @ np.ones(n)

    P, L, U = lu_factorization(A)
    x = lu_solve(A, b)

    fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
    fig.suptitle('LU Factorization with Partial Pivoting',
                 fontsize=17, fontweight='bold', y=1.04)

    titles = ['Original matrix A', 'Permutation P',
              'Lower triangular L', 'Upper triangular U']

    for ax, matrix, title in zip(axes, [A, P, L, U], titles):
        image = ax.imshow(matrix, cmap='coolwarm', aspect='equal')
        ax.set_title(title, fontsize=11, pad=12)
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.tick_params(labelsize=8)

        for i in range(n):
            for j in range(n):
                ax.text(j, i, f'{matrix[i,j]:.2f}',
                        ha='center', va='center', fontsize=7)

        fig.colorbar(image, ax=ax, fraction=0.046,
                     pad=0.04, shrink=0.75)

    fig.tight_layout(w_pad=2.0)
    plt.show()

    factor_residual = np.linalg.norm(P @ A - L @ U)
    solve_residual = np.linalg.norm(A @ x - b)
    relative_residual = solve_residual / (
        np.linalg.norm(A) * np.linalg.norm(x) + np.linalg.norm(b)
    )

    print(f'Factorization residual: {factor_residual:.3e}')
    print(f'Solution residual:      {solve_residual:.3e}')
    print(f'Relative residual:      {relative_residual:.3e}')
    print(f'Solution error:         {np.linalg.norm(x-np.ones(n)):.3e}')

In [4]:
widgets.interact(
    show_lu,
    n=widgets.IntSlider(value=4, min=2, max=8, step=1,
                        description='Dimension'),
    seed=widgets.IntSlider(value=42, min=0, max=100, step=1,
                           description='Seed'),
    matrix_type=widgets.Dropdown(
        options=['Random', 'Symmetric positive definite',
                 'Zero initial pivot'],
        description='Matrix type',
    ),
);

interactive(children=(IntSlider(value=4, description='Dimension', max=8, min=2), IntSlider(value=42, descripti…